### Middleware

In [8]:
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv('OPENAI_API_KEY')
os.environ["GEMINI_API_KEY"]=os.getenv('GEMINI_API_KEY')
os.environ["GROQ_API_KEY"]=os.getenv('GROQ_API')

### Summarization 

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
agent = create_agent(
    model="gpt-4o", 
    checkpointer=InMemorySaver(), 
    middleware=[
        SummarizationMiddleware(
        model="gpt-4o",
        trigger=("messages", 10),
        keep=("messages", 4)
        )
    ]
)

In [15]:
config = {"configurable": {"thread_id": "test-1"}}

In [16]:
question=[
"What is 5+7?",
"What is 12-4?",
"What is 36?",
"What is 20/4?",
"What is 15+9?",
"What is 87?"
]
for q in question:
    res = agent.invoke({"messages":[HumanMessage(content=q)]}, config=config)
    print(f"Messages: {res}")
    print(f"Messsges:{len(res["messages"])}")
    for message in res["messages"]:
        print(message)

Messages: {'messages': [HumanMessage(content='What is 5+7?', additional_kwargs={}, response_metadata={}, id='64052d77-4e2a-477c-804c-097112af73e2'), AIMessage(content='5 + 7 equals 12.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 14, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_94cd41efba', 'id': 'chatcmpl-E0s9bvihrVTV7Lg53Ll3zzMIOkoH5', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f5758-d639-7ed0-b127-d80246a89bb4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 8, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'au

In [39]:
from langchain.tools import tool
@tool 
def search_hotels(city:str)->str:
    """Search hotal in city-return long use more tokens"""
    return f""" Hotel in {city}:
    1. Grand Hotel: 5 star hotal
    2.City Hotal: 4 star hotal
    3.Budget Hotal: 3 star hotal"""


agent =create_agent(  model="groq:qwen/qwen3-32b", 
    checkpointer=InMemorySaver(),
    tools=[search_hotels] , 
    middleware=[
        SummarizationMiddleware(
        model="gpt-4o",
        trigger=("tokens", 450),
        keep=("tokens", 200)
        )
    ])

In [40]:
def countToken(messages):
    total_char = sum(len(str(n.content)) for n in messages)
    return total_char//4 

In [41]:
config = {"configurable": {"thread_id": "test-2"}}

In [42]:
cities = ["Tokyo", "Paris", "New York", "London", "Sydney", "Cairo"]

for city in cities :
    res= agent.invoke({"messages":[HumanMessage(f"find hotels in {city}")]}, config=config)
    print(f"Messages: {res["messages"]} ")
    print(f"{city}: ~{countToken(res["messages"])} tokens,{len(res["messages"])} messages")

    

Messages: [HumanMessage(content='find hotels in Tokyo', additional_kwargs={}, response_metadata={}, id='78f5848e-a221-4592-b6e5-e9ae4392dfe7'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to find hotels in Tokyo. Let me check the available tools. There's a function called search_hotels that takes a city parameter. The description says it returns long results using more tokens. Since the user specified Tokyo, I need to call this function with the city set to Tokyo. I'll make sure the parameters are correctly formatted as JSON within the tool_call tags.\n", 'tool_calls': [{'id': 'm97y1bezb', 'function': {'arguments': '{"city":"Tokyo"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 155, 'total_tokens': 260, 'completion_time': 0.159975596, 'completion_tokens_details': {'reasoning_tokens': 79}, 'prompt_time': 0.006372332, 'prompt_tokens_details': None, 'queue_time': 0.00

### Fraction


In [43]:
from langchain.tools import tool
@tool 
def search_hotels(city:str)->str:
    """Search hotal in city-return long use more tokens"""
    return f""" Hotel in {city}:
    1. Grand Hotel: 5 star hotal
    2.City Hotal: 4 star hotal
    3.Budget Hotal: 3 star hotal"""


agent =create_agent(  model="groq:qwen/qwen3-32b", 
    checkpointer=InMemorySaver(),
    tools=[search_hotels] , 
    middleware=[
        SummarizationMiddleware(
        model="gpt-4o",
        trigger=("fraction", 0.005),
        keep=("fraction", 0.002)
        )
    ])

In [44]:
cities = ["Tokyo", "Paris", "New York", "London", "Sydney", "Cairo"]

for city in cities :
    res= agent.invoke({"messages":[HumanMessage(f"find hotels in {city}")]}, config=config)
    print(f"Messages: {res["messages"]} ")
    print(f"{city}: ~{countToken(res["messages"])} tokens,{len(res["messages"])} messages")

    

Messages: [HumanMessage(content='find hotels in Tokyo', additional_kwargs={}, response_metadata={}, id='7e156f96-6adf-4895-a7bd-8b1693c03092'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to find hotels in Tokyo. Let me check the available tools. There\'s a function called search_hotels that takes a city parameter. The city here is Tokyo, so I need to call that function with "Tokyo" as the argument. I\'ll make sure the JSON is correctly formatted with the city name in the parameters. No other parameters are needed since the function only requires the city. Let me double-check the syntax to avoid any errors.\n', 'tool_calls': [{'id': '1gckan6vb', 'function': {'arguments': '{"city":"Tokyo"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 121, 'prompt_tokens': 155, 'total_tokens': 276, 'completion_time': 0.205026354, 'completion_tokens_details': {'reasoning_tokens': 95}, 'prompt_time': 0.006

### Human in the look

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage


In [3]:
def read_email_tool(email_id:str)->str:
    """Mock function to read email by its id"""
    return f"the content of email with {email_id}"
def send_email(recipient:str, subject:str, body :str)-> str:
    """Mock function for sending email."""
    return f"Email is sent to {recipient} with subject: {subject} and body: {body}"

In [33]:
agent= create_agent(model="groq:qwen/qwen3-32b",
                    tools=[read_email_tool,send_email], 
                    checkpointer=InMemorySaver(), 
                    middleware=[
                    

                     HumanInTheLoopMiddleware(
            interrupt_on={
                # Pass a dictionary with the allowed_decisions key, not a raw list
                "send_email": {"allowed_decisions": ["approve", "edit", "reject"]},
                "read_email_tool": False
            }
        )
                    
                    
                    
                    ] )

In [34]:
config={"configurable":{"thread_id":"teast-approve"}}

In [35]:
result=agent.invoke({"messages":[HumanMessage(content="send email to subhanatwork22@gmail.com with subject Hello and body 'how are you?' ")]} ,config=config )

In [36]:
result

{'messages': [HumanMessage(content="send email to subhanatwork22@gmail.com with subject Hello and body 'how are you?' ", additional_kwargs={}, response_metadata={}, id='9a6310e1-f816-4da0-9957-7ea86cf567d2'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to subhanatwork22@gmail.com with the subject "Hello" and the body "how are you?". Let me check the available functions.\n\nLooking at the tools provided, there\'s a send_email function. It requires recipient, subject, and body. The parameters are all there: recipient is the email address provided, subject is "Hello", and body is "how are you?". \n\nI need to make sure all required parameters are included. The function\'s required fields are recipient, subject, and body. The user provided all three, so that\'s covered. \n\nNo other functions are needed here. The read_email_tool isn\'t necessary because the user isn\'t asking to read an email. So the correct tool to use is send_ema

In [21]:
from langgraph.types import Command
if "__interrupt__" in result:
    result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {"type": "approve"}
            ]
        }
    ),
    config=config
)

In [23]:
result

{'messages': [HumanMessage(content="write a email to subhanatwork22@gmail.com with subject Hello and body 'how are you?' ", additional_kwargs={}, response_metadata={}, id='0ec56118-fce3-4f43-a028-97e1e8007dab'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to subhanatwork22@gmail.com with the subject "Hello" and body "how are you?". Let me check the available tools. There\'s the send_email function. It requires recipient, subject, and body. All three are provided here. The parameters are all strings. I need to structure the JSON with those parameters. Make sure the arguments match the required fields. No need to use the read_email_tool here since the user is sending, not reading. So the correct tool is send_email. Just fill in the parameters as specified.\n', 'tool_calls': [{'id': 'ndgm4qnbx', 'function': {'arguments': '{"body":"how are you?","recipient":"subhanatwork22@gmail.com","subject":"Hello"}', 'name': 'send_email'}, 'typ

### Reject

In [29]:
if "__interrupt__" in result:
    result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {"type": "reject"}
            ]
        }
    ),
    config=config
)

In [30]:
result

{'messages': [HumanMessage(content="send email to subhanatwork22@gmail.com with subject Hello and body 'how are you?' ", additional_kwargs={}, response_metadata={}, id='01c04496-74f6-4a5d-81d7-05e3bfec7f70'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to subhanatwork22@gmail.com with the subject "Hello" and the body "how are you?". Let me check the available functions. There\'s a send_email function that requires recipient, subject, and body. I need to make sure all three are included. The parameters are there: recipient is the email address provided, subject is "Hello", and body is "how are you?". I\'ll structure the tool call with these arguments. No need for the read_email_tool here since it\'s a send request. Just call send_email with the correct parameters.\n', 'tool_calls': [{'id': '0qmggjjnv', 'function': {'arguments': '{"body":"how are you?","recipient":"subhanatwork22@gmail.com","subject":"Hello"}', 'name': 'send_emai

In [37]:
if "__interrupt__" in result:
    result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
            "type": "edit",
            "edited_action": {
                "name": "send_email",
                "args": {"recipient":"Subhan@gmail.com", "subject":"Greeting", "body" :"Hi"}
            }}]}),config=config)
result

{'messages': [HumanMessage(content="send email to subhanatwork22@gmail.com with subject Hello and body 'how are you?' ", additional_kwargs={}, response_metadata={}, id='9a6310e1-f816-4da0-9957-7ea86cf567d2'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to subhanatwork22@gmail.com with the subject "Hello" and the body "how are you?". Let me check the available functions.\n\nLooking at the tools provided, there\'s a send_email function. It requires recipient, subject, and body. The parameters are all there: recipient is the email address provided, subject is "Hello", and body is "how are you?". \n\nI need to make sure all required parameters are included. The function\'s required fields are recipient, subject, and body. The user provided all three, so that\'s covered. \n\nNo other functions are needed here. The read_email_tool isn\'t necessary because the user isn\'t asking to read an email. So the correct tool to use is send_ema